In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import statsmodels.api as sm                         
from statsmodels.stats.outliers_influence import variance_inflation_factor  
import warnings
warnings.filterwarnings('ignore')                  

## Reading and data preparation

In [2]:
# Load activation dataset from CSV file
data = pd.read_csv("../activations_data.csv")
data.head()

,Id,Activo,Segmento,Flujo,SO,Bateria,Experimento
0,502805509,0,Orgánico,Wallet,NaN,56.84,Journey
1,38522610,1,Orgánico,Insurtech,Android,45.66,Journey
2,177366519,0,Orgánico,Wallet,Android,69.08,MLD
3,117814437,0,Orgánico,Wallet,Android,63.80,Journey
4,526560358,0,Orgánico,Cripto,Android,73.86,Journey


In [3]:
# Examine dataset structure and data types
data.info()

# Note: Segment, flow, SO, battery have null values, but we will not drop them because they are important for the analysis# Let's check if they may be imputed with the mean or median, but first we will check the distribution of the data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Id           500000 non-null  int64  
 1   Activo       500000 non-null  int64  
 2   Segmento     490030 non-null  object 
 3   Flujo        489952 non-null  object 
 4   SO           490048 non-null  object 
 5   Bateria      489943 non-null  float64
 6   Experimento  500000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 26.7+ MB


In [4]:
print("Original Segmento distribution:")
print(data["Segmento"].value_counts())

# Standardize category names and fill missing valuesprint(data["Segmento"].value_counts())

data["Segmento"] = data["Segmento"].replace({"Ecosistemic" : "Ecosistémico", "Organic" : "Orgánico"}).fillna("Unknown")
print("\nCleaned Segmento distribution:")

Original Segmento distribution:
Segmento
Ecosistémico    237043
Orgánico        214818
Organic          30177
Ecosistemic       7992
Name: count, dtype: int64

Cleaned Segmento distribution:


In [5]:
print("Original Flujo distribution:")
print(data["Flujo"].value_counts())

# Fill missing values with 'Unknown' categoryprint(data["Flujo"].value_counts())

data["Flujo"] = data["Flujo"].fillna("Unknown")
print("\nCleaned Flujo distribution:")

Original Flujo distribution:
Flujo
Insurtech       98268
Credits         98215
Account Fund    98157
Wallet          97781
Cripto          97531
Name: count, dtype: int64

Cleaned Flujo distribution:


In [6]:
print("Original SO distribution:")
print(data["SO"].value_counts())
# Fill missing values with 'Unknown' category

data["SO"] = data["SO"].fillna("Unknown")

print("\nCleaned SO distribution:")
print(data["SO"].value_counts())

Original SO distribution:
SO
Android    445390
iOS         44658
Name: count, dtype: int64

Cleaned SO distribution:
SO
Android    445390
iOS         44658
Unknown      9952
Name: count, dtype: int64


In [7]:
print("Original Bateria statistics:")
print(data["Bateria"].describe())
# Impute missing values with mean (appropriate for symmetric distribution)

data["Bateria"] = data["Bateria"].fillna(data["Bateria"].mean())

print("\nBateria statistics after imputation:")
print(data["Bateria"].describe())

Original Bateria statistics:
count    489943.000000
mean         60.001735
std           9.998365
min          11.150000
25%          53.260000
50%          60.010000
75%          66.730000
max         106.640000
Name: Bateria, dtype: float64

Bateria statistics after imputation:
count    500000.000000
mean         60.001735
std           9.897301
min          11.150000
25%          53.420000
50%          60.001735
75%          66.570000
max         106.640000
Name: Bateria, dtype: float64


In [8]:
# Explore final data structure after cleaning
print("Dataset shape:", data.shape)
print("\nAvailable columns:")
print(data.columns.tolist())
print("\nDataset information:")
data.info()
print("\nFirst few rows:")
data.head()

Dataset shape: (500000, 7)

Available columns:
['Id', 'Activo', 'Segmento', 'Flujo', 'SO', 'Bateria', 'Experimento']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Id           500000 non-null  int64  
 1   Activo       500000 non-null  int64  
 2   Segmento     500000 non-null  object 
 3   Flujo        500000 non-null  object 
 4   SO           500000 non-null  object 
 5   Bateria      500000 non-null  float64
 6   Experimento  500000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 26.7+ MB

First few rows:


,Id,Activo,Segmento,Flujo,SO,Bateria,Experimento
0,502805509,0,Orgánico,Wallet,Unknown,56.84,Journey
1,38522610,1,Orgánico,Insurtech,Android,45.66,Journey
2,177366519,0,Orgánico,Wallet,Android,69.08,MLD
3,117814437,0,Orgánico,Wallet,Android,63.80,Journey
4,526560358,0,Orgánico,Cripto,Android,73.86,Journey


## Exploratory analysis 

In [9]:
import plotly.express as px

In [21]:
data_graph = data.groupby(["Experimento", "Segmento"])["Activo"].mean().reset_index()
data_graph = data_graph.pivot(index="Experimento", columns="Segmento", values="Activo")
fig = px.imshow(data_graph, text_auto=True)
fig.show()

The organic segment shows the highest activation rate. The MLD experiment enhances its performance more effectively than the journey approach.

In [22]:
data_graph = data.groupby(["Experimento", "Flujo"])["Activo"].mean().reset_index()
data_graph = data_graph.pivot(index="Experimento", columns="Flujo", values="Activo")
fig = px.imshow(data_graph, text_auto=True)
fig.show()

**Simpson’s paradox**, also known as a confounding effect, is a statistical phenomenon in which a trend observed across several groups disappears or reverses when the groups are combined. This occurs due to the presence of a confounding variable that influences both the independent and dependent variables, potentially leading to misleading conclusions if not properly accounted for.

In this case, based on the graph, it appears that the journey approach achieves higher activation rates when the data are analyzed separately by experiment.

In [23]:
data_graph = data.groupby(["Experimento", "SO"])["Activo"].mean().reset_index()
data_graph = data_graph.pivot(index="Experimento", columns="SO", values="Activo")
fig = px.imshow(data_graph, text_auto=True)
fig.show()

As it above, **Simpson's paradox** occurs

In [24]:
data_graph = data.groupby(["Segmento", "SO"])["Activo"].mean().reset_index()
data_graph = data_graph.pivot(index="Segmento", columns="SO", values="Activo")
fig = px.imshow(data_graph, text_auto=True)
fig.show()

examining relationships between control variables, but there are not. 

## Evaluating statistical significance of the experiment 

In [10]:
# Analysis of response variable 'Activo' (Active/Activation status)
if 'Activo' in data.columns:
    print("Distribution of 'Activo' variable:")
    print(data['Activo'].value_counts())
    print(f"\nProportion: {data['Activo'].value_counts(normalize=True)}")

else:
    print("Variable 'Activo' not found in dataset.")
    print("Available variables:", data.columns.tolist())

Distribution of 'Activo' variable:
Activo
0    378237
1    121763
Name: count, dtype: int64

Proportion: Activo
0    0.756474
1    0.243526
Name: proportion, dtype: float64


In [11]:
# Data preparation for logistic regression modeling
# Exclude 'Id' and 'Activo' to obtain predictor variables
columns_to_exclude = ['Id', 'Activo']  # Target and identifier variables to exclude
available_columns = [col for col in columns_to_exclude if col in data.columns]
print(f"Columns found for exclusion: {available_columns}")

# Identify predictor variables (all columns except target and ID)
if 'Activo' in data.columns:
    predictor_columns = [col for col in data.columns if col not in columns_to_exclude]
    
    print(f"Available predictor variables: {predictor_columns}")
    print(f"Total number of predictors: {len(predictor_columns)}")
    
    # Classify variables by data type for appropriate preprocessing
    numeric_cols = data[predictor_columns].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = data[predictor_columns].select_dtypes(include=['object']).columns.tolist()
    
    print(f"\nNumeric variables ({len(numeric_cols)}): {numeric_cols}")
    print(f"Categorical variables ({len(categorical_cols)}): {categorical_cols}")
    
    # Check for missing values in predictor variables
    missing_data = data[predictor_columns].isnull().sum()
    print(f"\nMissing values by variable:")
    print(missing_data[missing_data > 0])
else:
    print("Cannot proceed: 'Activo' variable not found.")

Columns found for exclusion: ['Id', 'Activo']
Available predictor variables: ['Segmento', 'Flujo', 'SO', 'Bateria', 'Experimento']
Total number of predictors: 5

Numeric variables (1): ['Bateria']
Categorical variables (4): ['Segmento', 'Flujo', 'SO', 'Experimento']

Missing values by variable:
Series([], dtype: int64)


In [12]:
# Categorical variable encoding and final data preparation
if 'Activo' in data.columns:
    # Create a copy of dataframe for modeling to preserve original data
    model_data = data.copy()
    
    # Encode categorical variables using one-hot encoding (dummy variables)
    if len(categorical_cols) > 0:
        print("Encoding categorical variables...")
        # drop_first=True removes one category to avoid multicollinearity (reference category)
        categorical_dummies = pd.get_dummies(model_data[categorical_cols], prefix=categorical_cols, drop_first=True, dtype=int)
        print(f"Dummy variables created: {categorical_dummies.columns.tolist()}")
        
        # Combine numeric variables with dummy variables
        X = pd.concat([model_data[numeric_cols], categorical_dummies], axis=1)
    else:
        # If no categorical variables, use only numeric variables
        X = model_data[numeric_cols].copy()
    
    # Define response variable
    y = model_data['Activo']

else:    print("Cannot proceed without 'Activo' variable.")

Encoding categorical variables...
Dummy variables created: ['Segmento_Orgánico', 'Segmento_Unknown', 'Flujo_Credits', 'Flujo_Cripto', 'Flujo_Insurtech', 'Flujo_Unknown', 'Flujo_Wallet', 'SO_Unknown', 'SO_iOS', 'Experimento_MLD']


In [13]:
# Logistic regression model fitting
if 'Activo' in data.columns and X is not None:
    # Add constant for intercept term
    X_with_const = sm.add_constant(X)
    
    # Fit logistic regression model using statsmodels for statistical inference
    print("Fitting logistic regression model...")
    logit_model = sm.Logit(y, X_with_const)
    
    try:
        # Fit the model with optimized solver
        result = logit_model.fit(method='lbfgs', maxiter=1000, disp=False)
        
        # Display model summary with statistical tests
        print("LOGISTIC REGRESSION MODEL SUMMARY")
        print("=" * 50)
        print(result.summary())
        
        # Additional model information and goodness-of-fit metrics
        print(f"\nLog-Likelihood: {result.llf:.4f}")
        print(f"AIC (Akaike Information Criterion): {result.aic:.4f}")
        print(f"BIC (Bayesian Information Criterion): {result.bic:.4f}")
        print(f"Pseudo R-squared (McFadden): {result.prsquared:.4f}")
        
    except Exception as e:
        print(f"Error fitting the model: {e}")
        print("This may be due to multicollinearity or convergence issues.")
        result = None
        
else:
    print("Cannot fit model: data not properly prepared.")

Fitting logistic regression model...
LOGISTIC REGRESSION MODEL SUMMARY
                           Logit Regression Results                           
Dep. Variable:                 Activo   No. Observations:               500000
Model:                          Logit   Df Residuals:                   499988
Method:                           MLE   Df Model:                           11
Date:                Tue, 05 May 2026   Pseudo R-squ.:                 0.02213
Time:                        22:29:22   Log-Likelihood:            -2.7141e+05
converged:                       True   LL-Null:                   -2.7756e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -1.6679      0.023    -73.699      0.000      -1.712      -1.624
Bateria              -0.0003      0.000

In [14]:
# Calculation of Odds Ratios and Risk Factors Analysis
if 'result' in locals() and result is not None:
    print("ODDS RATIOS AND RISK FACTORS ANALYSIS")
    print("=" * 55)
    
    # Identify reference categories for categorical variables
    print("\nREFERENCE CATEGORIES FOR CATEGORICAL VARIABLES:")
    print("-" * 50)
    
    if 'categorical_cols' in locals() and len(categorical_cols) > 0:
        for cat_col in categorical_cols:
            # Get all unique categories from original variable
            unique_values = sorted(data[cat_col].unique())
            reference_category = unique_values[0]  # drop_first=True uses first alphabetically as reference
            
            print(f"• {cat_col}: Reference = '{reference_category}'")
            print(f"  Dummy variables created: {[col for col in X.columns if col.startswith(f'{cat_col}_')]}")
    else:
        print("No categorical variables in the model")
    
    print(f"\nNOTE: Odds Ratios for dummy variables are interpreted relative to their reference category")
    
    # Calculate odds ratios (exponential of coefficients)
    odds_ratios = np.exp(result.params)
    
    # Calculate 95% confidence intervals
    conf_int = np.exp(result.conf_int())
    
    # Create comprehensive results DataFrame
    or_results = pd.DataFrame({
        'Coefficient': result.params,
        'Std_Error': result.bse,
        'z_value': result.tvalues,
        'p_value': result.pvalues,
        'Odds_Ratio': odds_ratios,
        'CI_95%_lower': conf_int[0],
        'CI_95%_upper': conf_int[1]
    })
    
    # Classify risk factors vs protective factors
    or_results['Interpretation'] = or_results['Odds_Ratio'].apply(
        lambda x: 'Risk Factor' if x > 1 else 'Protective Factor'
    )
    
    # Add statistical significance indicator
    or_results['Significant'] = or_results['p_value'] < 0.05
    
    # Display results ordered by odds ratio magnitude
    print("\n\nRESULTS ORDERED BY ODDS RATIO (Highest to Lowest):")
    print("=" * 60)
    display_results = or_results.sort_values('Odds_Ratio', ascending=False)
    
    for variable, row in display_results.iterrows():
        if variable != 'const':  # Exclude intercept term
            status = "***" if row['Significant'] else ""
            print(f"\n{variable} {status}")
            print(f"  Odds Ratio: {row['Odds_Ratio']:.4f}")
            print(f"  95% CI: [{row['CI_95%_lower']:.4f}, {row['CI_95%_upper']:.4f}]")
            print(f"  p-value: {row['p_value']:.4f}")
            print(f"  Interpretation: {row['Interpretation']}")
            
            if row['Odds_Ratio'] > 1:
                percent_change = ((row['Odds_Ratio'] - 1) * 100)
                print(f"  Increases activation odds by {percent_change:.1f}%")
            else:
                percent_change = ((1 - row['Odds_Ratio']) * 100)
                print(f"  Decreases activation odds by {percent_change:.1f}%")
    
    print(f"\n*** = Statistically significant (p < 0.05)")
    
else:
    print("No fitted model available to calculate odds ratios.")

ODDS RATIOS AND RISK FACTORS ANALYSIS

REFERENCE CATEGORIES FOR CATEGORICAL VARIABLES:
--------------------------------------------------
• Segmento: Reference = 'Ecosistémico'
  Dummy variables created: ['Segmento_Orgánico', 'Segmento_Unknown']
• Flujo: Reference = 'Account Fund'
  Dummy variables created: ['Flujo_Credits', 'Flujo_Cripto', 'Flujo_Insurtech', 'Flujo_Unknown', 'Flujo_Wallet']
• SO: Reference = 'Android'
  Dummy variables created: ['SO_Unknown', 'SO_iOS']
• Experimento: Reference = 'Journey'
  Dummy variables created: ['Experimento_MLD']

NOTE: Odds Ratios for dummy variables are interpreted relative to their reference category


RESULTS ORDERED BY ODDS RATIO (Highest to Lowest):

Segmento_Orgánico ***
  Odds Ratio: 2.2422
  95% CI: [2.2095, 2.2753]
  p-value: 0.0000
  Interpretation: Risk Factor
  Increases activation odds by 124.2%

Segmento_Unknown ***
  Odds Ratio: 1.6435
  95% CI: [1.5690, 1.7215]
  p-value: 0.0000
  Interpretation: Risk Factor
  Increases activatio

In [15]:
# Statistical Significance Tests for Model Parameters
if 'result' in locals() and result is not None:
    print("STATISTICAL SIGNIFICANCE TESTS")
    print("=" * 45)
    
    # 1. Individual Wald tests for each parameter
    print("\n1. INDIVIDUAL WALD TESTS (Ho: βi = 0)")
    print("-" * 45)
    
    wald_results = pd.DataFrame({
        'Variable': result.params.index,
        'Coefficient': result.params.values,
        'Std_Error': result.bse.values,
        'z_Statistic': result.tvalues.values,
        'p_value': result.pvalues.values
    })
    
    # Classify significance levels
    wald_results['Significance'] = pd.cut(
        wald_results['p_value'], 
        bins=[0, 0.001, 0.01, 0.05, 0.1, 1.0], 
        labels=['***', '**', '*', '.', 'n.s.'],
        right=False
    )
    
    print(wald_results.to_string(index=False, float_format='%.4f'))
    print("\nSignificance codes: *** p<0.001, ** p<0.01, * p<0.05, . p<0.1, n.s. not significant")
    
    # 2. Likelihood Ratio Test for overall model significance
    print(f"\n\n2. GLOBAL LIKELIHOOD RATIO TEST")
    print("-" * 45)
    print(f"Model Log-Likelihood: {result.llf:.4f}")
    print(f"Null Log-Likelihood: {result.llnull:.4f}")
    print(f"LR Statistic: {result.llr:.4f}")
    print(f"LR test p-value: {result.llr_pvalue:.6f}")
    
    if result.llr_pvalue < 0.05:
        print("✓ Model is globally significant (p < 0.05)")
    else:
        print("✗ Model is NOT globally significant (p ≥ 0.05)")
    
    # 3. Summary of significant variables
    print(f"\n\n3. SIGNIFICANCE SUMMARY")
    print("-" * 30)
    
    significant_vars = wald_results[wald_results['p_value'] < 0.05]
    not_significant_vars = wald_results[wald_results['p_value'] >= 0.05]
    
    print(f"Statistically significant variables ({len(significant_vars)}):")
    for _, row in significant_vars.iterrows():
        if row['Variable'] != 'const':
            print(f"  • {row['Variable']} (p = {row['p_value']:.4f})")
    
    print(f"\nNON-significant variables ({len(not_significant_vars)}):")
    for _, row in not_significant_vars.iterrows():
        if row['Variable'] != 'const':
            print(f"  • {row['Variable']} (p = {row['p_value']:.4f})")
            
else:
    print("No fitted model available to perform significance tests.")

STATISTICAL SIGNIFICANCE TESTS

1. INDIVIDUAL WALD TESTS (Ho: βi = 0)
---------------------------------------------
         Variable  Coefficient  Std_Error  z_Statistic  p_value Significance
            const      -1.6679     0.0226     -73.6987   0.0000          ***
          Bateria      -0.0003     0.0003      -0.9237   0.3556         n.s.
Segmento_Orgánico       0.8074     0.0075     107.8245   0.0000          ***
 Segmento_Unknown       0.4968     0.0237      20.9976   0.0000          ***
    Flujo_Credits       0.0152     0.0106       1.4271   0.1535         n.s.
     Flujo_Cripto       0.0122     0.0107       1.1415   0.2537         n.s.
  Flujo_Insurtech       0.0033     0.0107       0.3085   0.7577         n.s.
    Flujo_Unknown       0.0154     0.0247       0.6251   0.5319         n.s.
     Flujo_Wallet       0.0070     0.0107       0.6536   0.5134         n.s.
       SO_Unknown       0.0092     0.0239       0.3839   0.7010         n.s.
           SO_iOS       0.0041     0.

In [16]:
# Model Diagnostics and Evaluation
if 'result' in locals() and result is not None:
    print("MODEL DIAGNOSTICS")
    print("=" * 30)
    
    # 1. Goodness of Fit Assessment
    print("\n1. GOODNESS OF FIT")
    print("-" * 20)
    
    # Model predictions
    y_pred_proba = result.predict(X_with_const)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    # Confusion matrix
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(y, y_pred)
    
    print("Confusion Matrix:")
    print(f"                Predicted")
    print(f"            0       1")
    print(f"Actual  0   {cm[0,0]:3d}   {cm[0,1]:3d}")
    print(f"        1   {cm[1,0]:3d}   {cm[1,1]:3d}")
    
    # Classification metrics
    accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
    precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
    recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
    
    print(f"\nClassification Metrics:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Sensitivity (Recall): {recall:.4f}")
    
    # AUC-ROC calculation
    try:
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y, y_pred_proba)
        print(f"  AUC-ROC: {auc:.4f}")
    except:
        print("  AUC-ROC: Not computable")
    
    # 2. Multicollinearity Assessment (VIF)
    print(f"\n\n2. MULTICOLLINEARITY ASSESSMENT (VIF)")
    print("-" * 40)
    
    try:
        # Calculate VIF only for original numeric variables
        if len(numeric_cols) > 1:
            X_numeric = X[numeric_cols]
            vif_data = pd.DataFrame()
            vif_data["Variable"] = X_numeric.columns
            vif_data["VIF"] = [variance_inflation_factor(X_numeric.values, i) 
                              for i in range(len(X_numeric.columns))]
            
            print("Variance Inflation Factor (VIF):")
            
            for _, row in vif_data.iterrows():
                vif_interpretation = ""
                if row['VIF'] > 10:
                    vif_interpretation = " (⚠️ Severe multicollinearity)"
                elif row['VIF'] > 5:
                    vif_interpretation = " (⚠️ Moderate multicollinearity)"
                elif row['VIF'] > 2:
                    vif_interpretation = " (⚠️ Mild multicollinearity)"
                else:
                    vif_interpretation = " (✓ No multicollinearity)"
                
                print(f"  {row['Variable']}: {row['VIF']:.2f}{vif_interpretation}")
        else:
            print("Insufficient numeric variables to calculate VIF")
            
    except Exception as e:
        print(f"Error calculating VIF: {e}")
    
    
else:
    print("No fitted model available for diagnostics.")

MODEL DIAGNOSTICS

1. GOODNESS OF FIT
--------------------
Confusion Matrix:
                Predicted
            0       1
Actual  0   378237     0
        1   121763     0

Classification Metrics:
  Accuracy: 0.7565
  Precision: 0.0000
  Sensitivity (Recall): 0.0000
  AUC-ROC: 0.5999


2. MULTICOLLINEARITY ASSESSMENT (VIF)
----------------------------------------
Insufficient numeric variables to calculate VIF


## Conclusions 
When introducing a new tool within a banking system, it is common to observe a **degree of resistance** to change. Financial behavior is inherently sensitive, and users tend to be risk-averse when it comes to managing their money. If the functionality of a new, more digital solution is not fully clear, it may fail to generate the level of trust required for users to complete activation.

In this context, users coming from an organic system are often more receptive. Because they already have prior exposure to the brand—its communication, value proposition, and overall experience—they are more likely to trust the new tool and engage with it. By contrast, users entering through a broader ecosystem may lack this familiarity. Even if they are successfully acquired, the absence of context and trust can reduce their likelihood of activation.

Additionally, considering that the experiment was based on a **machine learning-driven (MLD) strategy**, it is not surprising that a more acquisition-focused approach shows a higher probability of driving activations compared to a journey-based strategy. MLD approaches are typically optimized for short-term conversion signals, making them effective at identifying users with a higher immediate propensity to activate.

However, it is important not to evaluate performance solely based on activation volume. A more comprehensive assessment should include customer **quality metrics**, such as revenue generated over time, retention, or lifetime value. While an MLD strategy may drive a higher number of activations, the long-term value of those users may differ significantly from those acquired through a more structured journey. In some cases, fewer but more qualified users can generate greater overall value.

**In summary**, higher activation rates in organic systems can be explained by greater trust and familiarity, while MLD-driven strategies may boost short-term conversions. Nevertheless, a robust evaluation should balance both activation performance and downstream value to ensure sustainable impact.